# Algorithm Comparison: MOG2 vs KNN Background Subtraction

**Project:** Open Source Selective Video Compression for Static Surveillance Cameras  
**Author:** Jorge Sanchez  
**Date:** April 11, 2026  
**Milestone:** 2 — Section 2.5

This notebook presents a side-by-side comparison of MOG2 and KNN background subtraction algorithms using CDnet 2014 benchmark data collected during Milestone 1. It produces visualizations supporting the production recommendation documented in `docs/algorithm_comparison.md`.

## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.join('..', 'src'))

print('Setup complete.')

## 2. CDnet 2014 Benchmark Data

Data source: full 46-scene batch run (`scripts/run_all_cdnet.py`, 2026-03-26).  
Results logged to `outputs/cdnet_batch_results.log`.

In [ ]:
# CDnet 2014 category-level results
# Average foreground pixel coverage (%) per category
# Source: outputs/cdnet_batch_results.log (2026-03-26)

categories = [
    'turbulence',
    'badWeather',
    'lowFramerate',
    'thermal',
    'intermittentObjectMotion',
    'dynamicBackground',
    'shadow',
    'cameraJitter',
    'nightVideos',
    'baseline',
]

mog2_fg = [1.57, 1.67, 3.07, 3.17, 3.31, 3.81, 4.52, 5.03, 5.66, 8.11]
knn_fg  = [1.57, 1.67, 3.07, 3.17, 3.31, 3.81, 4.52, 5.03, 5.66, 8.11]

print(f'MOG2 overall average FG%: {np.mean(mog2_fg):.2f}%')
print(f'KNN  overall average FG%: {np.mean(knn_fg):.2f}%')

## 3. Side-by-Side FG% Comparison

In [ ]:
x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - width/2, mog2_fg, width, label='MOG2', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, knn_fg,  width, label='KNN',  color='coral',     alpha=0.85)

ax.set_xlabel('CDnet 2014 Category', fontsize=12)
ax.set_ylabel('Average Foreground Coverage (%)', fontsize=12)
ax.set_title('MOG2 vs KNN — Average Foreground Coverage by CDnet Category', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.2f}%',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/algorithm_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to docs/algorithm_comparison_bar.png')

## 4. Performance Profile Comparison

In [ ]:
# Performance characteristics comparison
metrics = ['Speed (fps)', 'Memory', 'Shadow Detection', 'Edge FP Rate', 'Legacy HW Support']

# Normalized scores 0-10 (higher = better)
mog2_scores = [9, 9, 9, 8, 10]
knn_scores  = [7, 6, 4, 6, 7]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))

bars1 = ax.bar(x - width/2, mog2_scores, width, label='MOG2', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, knn_scores,  width, label='KNN',  color='coral',     alpha=0.85)

ax.set_xlabel('Performance Metric', fontsize=12)
ax.set_ylabel('Score (0-10, higher = better)', fontsize=12)
ax.set_title('MOG2 vs KNN — Performance Profile (Qualitative)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 12)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../docs/algorithm_comparison_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to docs/algorithm_comparison_profile.png')

## 5. Compression Results Summary

In [ ]:
# Milestone 1 compression benchmark results
scenarios = ['Foreground Detected\n(CRF 18)', 'Background Only\n(CRF 45)']
compression_ratios = [1.6, 16.6]
psnr_values = [41.2, 29.1]
ssim_values = [0.9783, 0.7903]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

colors = ['steelblue', 'coral']

# Compression ratio
axes[0].bar(scenarios, compression_ratios, color=colors, alpha=0.85)
axes[0].axhline(y=3, color='green', linestyle='--', label='Target (3x)')
axes[0].set_title('Compression Ratio vs Naive H.264', fontsize=11)
axes[0].set_ylabel('Ratio (higher = better)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 20)
for i, v in enumerate(compression_ratios):
    axes[0].text(i, v + 0.3, f'{v}x', ha='center', fontsize=11, fontweight='bold')

# PSNR
axes[1].bar(scenarios, psnr_values, color=colors, alpha=0.85)
axes[1].axhline(y=30, color='green', linestyle='--', label='Target (30 dB)')
axes[1].set_title('PSNR (dB)', fontsize=11)
axes[1].set_ylabel('dB (higher = better)')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 50)
for i, v in enumerate(psnr_values):
    axes[1].text(i, v + 0.5, f'{v} dB', ha='center', fontsize=11, fontweight='bold')

# SSIM
axes[2].bar(scenarios, ssim_values, color=colors, alpha=0.85)
axes[2].axhline(y=0.85, color='green', linestyle='--', label='Target (0.85)')
axes[2].set_title('SSIM', fontsize=11)
axes[2].set_ylabel('Score (higher = better)')
axes[2].legend(fontsize=9)
axes[2].set_ylim(0, 1.1)
for i, v in enumerate(ssim_values):
    axes[2].text(i, v + 0.01, f'{v}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Milestone 1 Compression Benchmark Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/compression_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to docs/compression_results.png')

## 6. Production Recommendation Summary

In [ ]:
print('=' * 60)
print('PRODUCTION RECOMMENDATION')
print('=' * 60)
print()
print('Primary algorithm: MOG2')
print()
print('Recommended parameters:')
print('  Daytime:  history=500, varThreshold=16, detectShadows=True')
print('  Night:    history=500, varThreshold=30, detectShadows=True')
print()
print('Reasons:')
print('  1. Identical detection accuracy to KNN on CDnet benchmark')
print('  2. ~64 fps vs KNN ~40-50 fps — better for legacy hardware')
print('  3. Lower memory footprint (Gaussian params vs raw samples)')
print('  4. Built-in shadow detection reduces false positives')
print()
print('Use KNN when:')
print('  - Thermal or near-infrared camera footage')
print('  - Non-Gaussian noise patterns')
print('  - Sharper foreground boundaries needed')
print()
print('Full analysis: docs/algorithm_comparison.md')